%md
#  Monitoramento de APIs dos Departamentos Regionais (DRs)

##  Objetivo

Este notebook realiza o monitoramento da disponibilidade e integridade das APIs utilizadas para ingestão de dados dos Departamentos Regionais (DRs).

O processo executa chamadas automáticas para todos os endpoints configurados e coleta métricas técnicas que permitem identificar:

- Disponibilidade das APIs
- Tempo de resposta (latência)
- Problemas de autenticação
- Problemas de certificado SSL
- Alterações na estrutura de dados retornada
- Bloqueios por firewall ou serviços de segurança
- APIs offline ou indisponíveis

---

##  Funcionamento do Monitoramento

Para cada **DR configurado**, o processo executa as seguintes etapas:

1. Autenticação na API  
2. Consulta aos endpoints configurados  
3. Coleta de métricas da resposta  
4. Retry automático em caso de erro SSL  
5. Classificação automática de erros

---

##  Estrutura do Resultado

O monitoramento gera um DataFrame contendo uma linha para cada combinação de **DR e endpoint monitorado**.

| Coluna | Descrição |
|------|------|
| data_execucao | Data da execução do monitoramento |
| dr | Departamento Regional |
| endpoint | Endpoint monitorado |
| status_execucao | Resultado da execução |
| codigo_status | Código HTTP retornado |
| latencia_ms | Tempo de resposta da API em milissegundos |
| quantidade_registros | Quantidade de registros retornados |
| quantidade_colunas | Quantidade de colunas identificadas |
| tamanho_resposta_bytes | Tamanho da resposta em bytes |
| detalhe_retorno | Estrutura retornada ou mensagem de erro |
| ssl_inseguro | Indica se foi necessário ignorar validação SSL |

---

##  Classificação automática de erros

| Status | Significado |
|------|------|
| SUCESSO | API respondeu corretamente |
| BLOQUEIO_FIREWALL | Requisição bloqueada por firewall ou WAF |
| API_OFFLINE | API ou serviço indisponível |
| ERRO_SSL | Problema no certificado SSL |
| ERRO_HTTP | API respondeu com erro HTTP |
| ERRO_DESCONHECIDO | Erro não classificado |

---

##  Benefícios

Este monitoramento permite detectar rapidamente:

- APIs indisponíveis
- Degradação de performance
- Problemas de autenticação
- Certificados SSL inválidos
- Mudanças na estrutura de dados
- Bloqueios por firewall

In [0]:
# Bibliotecas padrão
import requests
import json
import time
import base64

# Controle de data/hora
from datetime import datetime

# Execução paralela para consultar múltiplos DRs ao mesmo tempo
from concurrent.futures import ThreadPoolExecutor, as_completed

# Tratamento de erros de conexão
from requests.exceptions import SSLError, ConnectTimeout, ReadTimeout

# Tipos de dados para criação do DataFrame no Spark
from pyspark.sql.types import *

In [0]:
var_adls_uri, notebook_params = conector
var_adf = notebook_params.var_adf
dls = notebook_params.var_dls
params = json.loads(dbutils.widgets.get("params").replace("\'", '\"'))
bot_name = dbutils.widgets.get("bot_name").replace("\'", '\"')

In [0]:
# Cache simples para armazenar tokens Bearer já autenticados
# Evita fazer login repetidamente para o mesmo DR
token_cache = {}
latencia_limite_ms = 5000

# Regiões (DRs) e endpoints monitorados
regions = params["regions"]
endpoints = params["endpoints"]

# Data da execução do monitoramento
now = datetime.now().strftime("%d/%m/%Y")

# Define qual KeyVault será usado para buscar as senhas
env = "dev"
scope = "scopedev"

try:
    env = json.loads(dbutils.widgets.get("env").replace("\'", '\"')).get("env")
except:
    pass

# Se for ambiente de produção usa o KeyVault de produção
if env == "prod":
    scope = "scopeprod"

In [0]:
def get_password(secret_key):

    """
    Recupera a senha armazenada no Azure KeyVault.
    Cada DR possui uma chave específica configurada no params.
    """

    try:
        return dbutils.secrets.get(
            scope=scope,
            key=secret_key
        )

    except Exception:
        raise Exception("SECRET_NOT_FOUND")

In [0]:
def build_url(region_conf, endpoint):

    """
    Constrói a URL da API combinando:
    - domínio do DR
    - path da API
    - endpoint
    """

    protocol = "https"

    origem = region_conf["url"].replace("https://", "").replace("http://", "")

    path = region_conf["path"]

    porta = region_conf["porta"]

    if porta:
        api_url = f"{protocol}://{origem}:{porta}/{path}/{endpoint}?count=false&limit=1&offset=0"
    else:
        api_url = f"{protocol}://{origem}/{path}/{endpoint}?count=false&limit=1&offset=0"

    return api_url

In [0]:
def authenticate(region_conf):

    """
    Realiza autenticação na API do DR.

    Tipos suportados:
    - Bearer Token (login via /user/login)
    - Basic Auth (username:password base64)

    Retorno:
    headers, status_code, error_message
    """

    auth_type = region_conf.get("auth", "bearer")

    username = region_conf["username"]

    cache_key = f"{region_conf['url']}_{auth_type}"

    # reutiliza token apenas para Bearer
    if auth_type == "bearer" and cache_key in token_cache:

        token = token_cache[cache_key]

        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }

        return headers, 200, None

    # busca senha no keyvault
    try:
        password = get_password(region_conf["password_key"])
    except Exception as e:
        return None, None, f"SECRET_ERROR: {str(e)}"

    # ===============================
    # BASIC AUTH
    # ===============================
    if auth_type == "basic":

        try:

            credentials = f"{username}:{password}"

            token = base64.b64encode(credentials.encode()).decode()

            headers = {
                "Authorization": f"Basic {token}",
                "Content-Type": "application/json"
            }

            return headers, 200, None

        except Exception as e:

            return None, None, f"BASIC_AUTH_ERROR: {str(e)}"

    # ===============================
    # BEARER AUTH
    # ===============================
    if auth_type == "bearer":
        porta = region_conf["porta"]
        if porta:
            login_url = f"{region_conf['url']}:{porta}/user/login"
        else:
            login_url = f"{region_conf['url']}/user/login"

        payload = {
            "username": username,
            "password": password
        }

        try:

            response = requests.post(
                login_url,
                json=payload,
                verify=True,
                timeout=60
            )

        except requests.exceptions.SSLError:

            # retry sem verificação SSL
            try:

                response = requests.post(
                    login_url,
                    json=payload,
                    verify=False,
                    timeout=60
                )

            except Exception as e:

                return None, None, f"LOGIN_CONNECTION_ERROR: {str(e)}"

        except requests.exceptions.ConnectTimeout:

            return None, None, "LOGIN_CONNECT_TIMEOUT"

        except requests.exceptions.ReadTimeout:

            return None, None, "LOGIN_READ_TIMEOUT"

        except Exception as e:

            return None, None, f"LOGIN_REQUEST_ERROR: {str(e)}"

        # API respondeu erro
        if response.status_code != 200:

            return None, response.status_code, f"LOGIN_HTTP_ERROR: {response.text[:300]}"

        try:

            data = response.json()

        except Exception:

            return None, response.status_code, "LOGIN_INVALID_JSON"

        token = data.get("token") or data.get("access_token")

        if not token:

            return None, response.status_code, "LOGIN_TOKEN_NOT_FOUND"

        # salva token no cache
        token_cache[cache_key] = token

        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }

        return headers, 200, None

    return None, None, "AUTH_TYPE_NOT_SUPPORTED"

In [0]:
def make_request(api_url, headers):

    """
    Executa chamada à API.

    Estratégia:
    1º tentativa → verify SSL = True
    se falhar SSL → tenta novamente com verify=False
    """

    ssl_inseguro = False

    try:

        response = requests.post(
            url=api_url,
            headers=headers,
            verify=True,
            timeout=60
        )

        return response, None, ssl_inseguro


    except requests.exceptions.SSLError as ssl_error:

        # tentativa sem verificação SSL
        try:

            ssl_inseguro = True

            response = requests.post(
                url=api_url,
                headers=headers,
                verify=False,
                timeout=60
            )

            return response, None, ssl_inseguro

        except Exception as retry_error:

            return None, str(retry_error), ssl_inseguro


    except requests.exceptions.ConnectTimeout:

        return None, "CONNECT_TIMEOUT", ssl_inseguro


    except requests.exceptions.ReadTimeout:

        return None, "READ_TIMEOUT", ssl_inseguro


    except requests.exceptions.ConnectionError as conn_error:

        return None, f"CONNECTION_ERROR: {str(conn_error)}", ssl_inseguro


    except Exception as e:

        return None, str(e), ssl_inseguro

In [0]:
# Função para classificar automaticamente o tipo de erro da API
def classify_error(response_text, error_msg, status_code):
    
    """
    Classifica automaticamente o tipo de erro encontrado na requisição.
    Permite identificar problemas de infraestrutura, autenticação e API.
    """

    text = ""

    if response_text:
        text = response_text.lower()

    if error_msg:
        text += " " + str(error_msg).lower()

    # ===============================
    # SECRET / KEYVAULT
    # ===============================
    if "secret" in text:
        return "ERRO_SECRET"

    # ===============================
    # FIREWALL / WAF
    # ===============================
    firewall_keywords = [
        "acesso bloqueado",
        "access denied",
        "blocked",
        "cloudflare",
        "gocache",
        "akamai",
        "request id",
        "security service"
    ]

    if any(k in text for k in firewall_keywords):
        return "BLOQUEIO_FIREWALL"

    # ===============================
    # API OFFLINE
    # ===============================
    offline_keywords = [
        "connection refused",
        "connecttimeout",
        "max retries exceeded",
        "timed out",
        "10061",
        "service unavailable",
        "name or service not known"
    ]

    if any(k in text for k in offline_keywords):
        return "API_OFFLINE"

    # ===============================
    # TLS / SNI
    # ===============================
    tls_keywords = [
        "tlsv1_unrecognized_name",
        "unrecognized name",
        "tls handshake"
    ]

    if any(k in text for k in tls_keywords):
        return "ERRO_TLS_SNI"

    # ===============================
    # SSL
    # ===============================
    if "ssl" in text:
        return "ERRO_SSL"

    # ===============================
    # ERROS HTTP
    # ===============================

    if status_code:

        if status_code == 401:
            return "AUTH_INVALIDA"

        if status_code == 403:
            return "ACESSO_NEGADO"

        if status_code == 404:
            return "ENDPOINT_NAO_ENCONTRADO"

        if status_code >= 500:
            return "API_ERRO_INTERNO"

        if status_code >= 400:
            return "ERRO_HTTP_CLIENT"

    return "ERRO_DESCONHECIDO"

In [0]:
def process_dr(dr, region_conf):

    """
    Executa o monitoramento para um DR específico:
    - autentica
    - consulta endpoints
    - coleta métricas
    - classifica erros automaticamente
    """

    results_local = []

    headers, auth_status, auth_error = authenticate(region_conf)

    if headers is None:

        erro_tipo = classify_error(None, auth_error, auth_status)

        results_local.append((
            now,
            dr,
            "AUTH",
            erro_tipo,
            auth_status,
            None,
            None,
            None,
            None,
            auth_error,
            None
        ))

        return results_local


    for endpoint_key, endpoint_label in endpoints.items():

        url = build_url(region_conf, endpoint_key)

        start_time = time.time()

        response, request_error, ssl_inseguro = make_request(url, headers)

        latencia_ms = round((time.time() - start_time) * 1000)


        # erro de conexão
        if response is None:

            erro_tipo = classify_error(None, request_error, None)

            results_local.append((
                now,
                dr,
                endpoint_label,
                erro_tipo,
                None,
                latencia_ms,
                None,
                None,
                None,
                request_error,
                ssl_inseguro
            ))

            continue


        status_code = response.status_code

        response_size_bytes = len(response.content)


        if status_code in [200, 201]:

            try:

                data = response.json()

                results_data = data.get("results") or data.get("data") or []

                quantidade_registros = len(results_data)

                if quantidade_registros > 0:

                    colunas = list(results_data[0].keys())

                    quantidade_colunas = len(colunas)

                    colunas_str = json.dumps(colunas)

                else:

                    quantidade_colunas = 0
                    colunas_str = "[]"

            except:

                quantidade_registros = None
                quantidade_colunas = None
                colunas_str = response.text[:500]

            if latencia_ms > latencia_limite_ms:
                status_execucao = "API_LENTA"
            else:
                status_execucao = "SUCESSO"

        else:

            quantidade_registros = None
            quantidade_colunas = None
            colunas_str = response.text[:500]

            status_execucao = classify_error(response.text, None, status_code)


        results_local.append((
            now,
            dr,
            endpoint_label,
            status_execucao,
            status_code,
            latencia_ms,
            quantidade_registros,
            quantidade_colunas,
            response_size_bytes,
            colunas_str,
            ssl_inseguro
        ))

    return results_local

In [0]:
# lista final de resultados
results = []

# número de threads dinâmico
max_threads = min(16, len(regions))

with ThreadPoolExecutor(max_workers=max_threads) as executor:

    future_to_dr = {
        executor.submit(process_dr, dr, region_conf): dr
        for dr, region_conf in regions.items()
    }

    for future in as_completed(future_to_dr):

        dr = future_to_dr[future]

        try:

            dr_result = future.result(timeout=120)

            results.extend(dr_result)

        except Exception as e:

            results.append((
                now,
                dr,
                "THREAD_ERROR",
                str(e),
                None,
                None,
                None,
                None,
                None,
                None,
                None
            ))

In [0]:
schema = StructType([
    StructField("data_execucao", StringType(), True),
    StructField("dr", StringType(), True),
    StructField("endpoint", StringType(), True),
    StructField("status_execucao", StringType(), True),
    StructField("codigo_status", IntegerType(), True),
    StructField("latencia_ms", IntegerType(), True),
    StructField("quantidade_registros", IntegerType(), True),
    StructField("quantidade_colunas", IntegerType(), True),
    StructField("tamanho_resposta_bytes", IntegerType(), True),
    StructField("detalhe_retorno", StringType(), True),
    StructField("ssl_inseguro", BooleanType(), True)
])
df = spark.createDataFrame(results, schema)

In [0]:
df.write.mode("append").format("delta").save("caminho")